In [0]:
-- [Week47] LOY-00567 - CineClub Anniversary Offers - Push - Oct 1
-- 500 SCENE+ pts for 2+ Lg fountain drinks in a single transaction, Oct 1–31, 2026
-- Rules: once per member; txn = Transaction_Number x LocationId x TransactionDateid;
--        excl. voucher-scanned drink lines and txns with SCENE+ points redeemed

-- ===== VIEW 1: push stages per member (ET) =====
CREATE OR REPLACE TEMP VIEW push_stage AS
WITH evt AS (
    SELECT 'sent' AS stage, user_id, external_user_id, time
    FROM cpx_dataanalytics_gold.marketing.mkt_braze_users_messages_pushnotification_send
    WHERE canvas_name = '[Week47] LOY-00567 - CineClub Anniversary Offers - Push - Oct 1'
    UNION ALL
    SELECT 'bounced', user_id, external_user_id, time
    FROM cpx_dataanalytics_gold.marketing.mkt_braze_users_messages_pushnotification_bounce
    WHERE canvas_name = '[Week47] LOY-00567 - CineClub Anniversary Offers - Push - Oct 1'
    UNION ALL
    SELECT 'opened', user_id, external_user_id, time
    FROM cpx_dataanalytics_gold.marketing.mkt_braze_users_messages_pushnotification_open
    WHERE canvas_name = '[Week47] LOY-00567 - CineClub Anniversary Offers - Push - Oct 1'
),
braze_map AS (                                -- Braze user -> guid (from send)
    SELECT user_id, MAX(external_user_id) AS guid
    FROM evt WHERE stage = 'sent'
    GROUP BY user_id
),
cust AS (
    SELECT DISTINCT cc_user_profile_guid, cde_id
    FROM cpx_dataanalytics_gold.customer360.cpx_d_customer
    WHERE cc_user_profile_guid IS NOT NULL
)
SELECT c.cde_id,
       MIN(CASE WHEN e.stage = 'sent'    THEN from_utc_timestamp(from_unixtime(e.time), 'America/New_York') END) AS sent_et,
       MIN(CASE WHEN e.stage = 'bounced' THEN from_utc_timestamp(from_unixtime(e.time), 'America/New_York') END) AS bounced_et,
       MIN(CASE WHEN e.stage = 'opened'  THEN from_utc_timestamp(from_unixtime(e.time), 'America/New_York') END) AS opened_et
FROM evt e
JOIN braze_map b ON b.user_id = e.user_id
JOIN cust c      ON c.cc_user_profile_guid = b.guid
GROUP BY c.cde_id;

-- ===== VIEW 2: eligible member-days (push recipients only) =====
CREATE OR REPLACE TEMP VIEW active_member_days AS
WITH cc_cde_ids AS (                          -- CineClub CDE_IDs in transactions (Oct)
    SELECT DISTINCT t.CDE_ID, t.VisitDateId
    FROM cpx_dataanalytics_gold.customer360.c360_f_transaction t
    WHERE t.VisitDateId BETWEEN 20261001 AND 20261031
      AND t.CDE_ID > 0
      AND t.CineClubFLAG = 'Yes'
),
cc_cde_status AS (                            -- plan_id + renewal_status (same-day snapshot)
    SELECT c.CDE_ID, c.VisitDateId, s.plan_id, s.renewal_status
    FROM cc_cde_ids c
    LEFT JOIN cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot s
           ON s.cde_id = c.CDE_ID
          AND s.snapshot_date_id = CAST(c.VisitDateId AS STRING)
)
SELECT a.*
FROM cc_cde_status a
JOIN push_stage p ON p.cde_id = a.CDE_ID AND p.sent_et IS NOT NULL
WHERE (a.plan_id IN ('annual1', 'gift-plan')
       AND a.renewal_status IN ('Active', 'Active_Non_Renewing', 'Active_Annual_Upgrade'))
   OR (a.plan_id = 'monthly1'
       AND a.renewal_status = 'Active_Annual_Upgrade');

-- ===== VIEW 3: one row per member (transaction grain) =====
CREATE OR REPLACE TEMP VIEW member_totals AS
WITH lg_drink_items AS (
    SELECT itemskey
    FROM cpx_dataanalytics_gold.customer360.cpx_d_item
    WHERE Item_ID IN ('93978','94119','14068','80765','93985','88085','93263',
                      '93264','93262','40279','40217','89592','94114')
),
txn_lines AS (                                -- ALL lines on eligible member-days (needed for txn-level exclusions)
    SELECT t.CDE_ID, t.TransactionDateid, t.LocationId, t.Transaction_Number,
           t.Transaction_DateTime, t.Quantity, t.SCENE_Points_Redeemed, a.plan_id,
           (d.itemskey IS NOT NULL AND t.ConcessionFLAG = 'Yes')  AS is_lg_drink,
           COALESCE(t.Voucher_ScannedFLAG, 'No') = 'Yes'          AS is_voucher
    FROM cpx_dataanalytics_gold.customer360.c360_f_transaction t
    JOIN active_member_days a   ON t.CDE_ID = a.CDE_ID AND t.VisitDateId = a.VisitDateId
    LEFT JOIN lg_drink_items d  ON t.ItemSkey = d.itemskey
    WHERE t.VisitDateId BETWEEN 20261001 AND 20261031
),
txn_drinks AS (                               -- txn = Transaction_Number x LocationId x TransactionDateid
    SELECT CDE_ID, TransactionDateid, LocationId, Transaction_Number,
           MAX(plan_id)                                                         AS plan_id,
           MIN(Transaction_DateTime)                                            AS txn_et,
           SUM(CASE WHEN is_lg_drink THEN Quantity ELSE 0 END)                  AS txn_lg_drinks,        -- all Lg drinks, net of refunds
           SUM(CASE WHEN is_lg_drink AND NOT is_voucher THEN Quantity ELSE 0 END) AS txn_lg_drinks_elig, -- excl. voucher lines
           MAX(CASE WHEN COALESCE(SCENE_Points_Redeemed, 0) > 0 THEN 1 ELSE 0 END) AS has_pts_redeemed
    FROM txn_lines
    GROUP BY CDE_ID, TransactionDateid, LocationId, Transaction_Number
    HAVING MAX(CASE WHEN is_lg_drink THEN 1 ELSE 0 END) = 1
),
txn_flagged AS (
    SELECT *, (txn_lg_drinks_elig >= 2 AND has_pts_redeemed = 0) AS is_qualifying
    FROM txn_drinks
)
SELECT x.CDE_ID,
       CASE MAX_BY(x.plan_id, x.txn_et)
            WHEN 'annual1'   THEN '1. Annual'
            WHEN 'gift-plan' THEN '2. Gift Plan'
            WHEN 'monthly1'  THEN '3. Monthly upgrading'
       END                                                             AS bucket,
       SUM(x.txn_lg_drinks)                                            AS lg_drinks_oct,
       MAX(CASE WHEN x.has_pts_redeemed = 0 THEN x.txn_lg_drinks_elig END) AS max_elig_lg_drinks_one_txn,
       MIN(CASE WHEN x.is_qualifying THEN x.txn_et END)                AS first_qualifying_txn_et,
       MAX(CASE WHEN x.is_qualifying AND p.opened_et IS NOT NULL AND x.txn_et >= p.opened_et
                THEN 1 ELSE 0 END)                                     AS qualified_after_open,
       MAX(p.opened_et)                                                AS opened_et
FROM txn_flagged x
JOIN push_stage p ON p.cde_id = x.CDE_ID
GROUP BY x.CDE_ID;

-- ===== VIEW 4: qualifiers (once per member, 500 pts) =====
CREATE OR REPLACE TEMP VIEW offer_qualifiers AS
SELECT CDE_ID, bucket, lg_drinks_oct, max_elig_lg_drinks_one_txn,
       first_qualifying_txn_et, opened_et,
       500 AS points_awarded,
       CASE WHEN opened_et IS NULL         THEN '3. Qualified, never opened push'
            WHEN qualified_after_open = 1  THEN '1. Qualified after opening'
            ELSE                                '2. Qualified before opening'
       END AS conversion_timing
FROM member_totals
WHERE first_qualifying_txn_et IS NOT NULL;

-- ===== OUTPUT A: bucket summary =====
SELECT COALESCE(bucket, 'Total')                                        AS bucket,
       COUNT(DISTINCT CDE_ID)                                           AS members_bought_lg_drink,
       SUM(lg_drinks_oct)                                               AS lg_drinks,
       COUNT(DISTINCT CASE WHEN first_qualifying_txn_et IS NOT NULL THEN CDE_ID END)       AS qualifiers,
       500 * COUNT(DISTINCT CASE WHEN first_qualifying_txn_et IS NOT NULL THEN CDE_ID END) AS points_awarded
FROM member_totals
GROUP BY ROLLUP(bucket)
ORDER BY bucket NULLS LAST;

-- ===== OUTPUT B: push funnel =====
WITH f AS (
  SELECT (SELECT COUNT(DISTINCT cde_id) FROM push_stage WHERE sent_et IS NOT NULL)                       AS sent,
         (SELECT COUNT(DISTINCT cde_id) FROM push_stage WHERE sent_et IS NOT NULL AND bounced_et IS NULL) AS received,
         (SELECT COUNT(DISTINCT cde_id) FROM push_stage WHERE sent_et IS NOT NULL
             AND opened_et < TIMESTAMP '2026-11-01 00:00:00')                                            AS opened,
         (SELECT COUNT(DISTINCT CDE_ID) FROM active_member_days)                                         AS visited_eligible,
         (SELECT COUNT(DISTINCT CDE_ID) FROM member_totals WHERE lg_drinks_oct > 0)                      AS bought_lg_drink,
         (SELECT COUNT(DISTINCT CDE_ID) FROM offer_qualifiers)                                           AS qualified,
         (SELECT COUNT(DISTINCT CDE_ID) FROM offer_qualifiers
             WHERE conversion_timing = '1. Qualified after opening')                                     AS qualified_after_open
)
SELECT '1. Push sent' AS stage, sent AS members, CAST(NULL AS DOUBLE) AS pct_of_prev, 100.00 AS pct_of_sent FROM f
UNION ALL SELECT '2. Received (not bounced)',         received,             ROUND(100 * received / sent, 2),                    ROUND(100 * received / sent, 2) FROM f
UNION ALL SELECT '3. Opened / tapped (in Oct)',       opened,               ROUND(100 * opened / received, 2),                  ROUND(100 * opened / sent, 2) FROM f
UNION ALL SELECT '4. Visited as eligible member',     visited_eligible,     ROUND(100 * visited_eligible / received, 2),        ROUND(100 * visited_eligible / sent, 2) FROM f
UNION ALL SELECT '5. Bought 1+ Lg drink',             bought_lg_drink,      ROUND(100 * bought_lg_drink / visited_eligible, 2), ROUND(100 * bought_lg_drink / sent, 2) FROM f
UNION ALL SELECT '6. Qualified (2+ Lg in one txn)', qualified,            ROUND(100 * qualified / bought_lg_drink, 2),        ROUND(100 * qualified / sent, 2) FROM f
UNION ALL SELECT '7. Qualified after opening',        qualified_after_open, ROUND(100 * qualified_after_open / opened, 2),      ROUND(100 * qualified_after_open / sent, 2) FROM f;

-- ===== OUTPUT C: conversion timing by bucket =====
SELECT COALESCE(bucket, 'Total')          AS bucket,
       COALESCE(conversion_timing, 'All') AS conversion_timing,
       COUNT(*)                           AS members,
       SUM(points_awarded)                AS points_awarded
FROM offer_qualifiers
GROUP BY ROLLUP(bucket, conversion_timing)
ORDER BY bucket NULLS LAST, conversion_timing NULLS LAST;

-- ===== OUTPUT D: CDE_ID list =====
SELECT CDE_ID, bucket, conversion_timing, points_awarded, first_qualifying_txn_et,
       max_elig_lg_drinks_one_txn, lg_drinks_oct, opened_et
FROM offer_qualifiers
ORDER BY bucket, first_qualifying_txn_et;